<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.3/blob/main/02_Dynamic_State_Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 02_Dynamic_State_Language.py
# ============================================================
#
# FINAL PUBLICATION VERSION
#
# INPUT
# -----
# 01_TD_NMR_descriptors.csv
#
# generated by:
# 01_TDNMR_descriptor_extraction.py
#
#
# INPUT DESCRIPTORS
# -----------------
# Canonical_ID
# ShortFraction
# MidFraction
# LongFraction
# Weighted_logmean_T2_ms
# Width_log10T2
# N_detected_peaks
# Entropy_norm
#
#
# OUTPUT
# ------
# 01_Dynamic_State_Language.csv
# 02_Dynamic_State_Language_rules.csv
# 03_Dynamic_State_Language_audit.csv
# 04_Dynamic_State_Language_embedding_384D.csv
# 05_Metadata.json
#
# SI OUTPUT
# ---------
# 06_Table_Sx_Dynamic_State_Language.xlsx
# Table_Sx_Dynamic_State_Language.pdf
# Table_Sx_Dynamic_State_Language.png
#
# ZIP
# ---
# 02_Dynamic_State_Language_output.zip
#
#
# IMPORTANT SCIENTIFIC PRINCIPLE
# ------------------------------
# X-side:
#   TD-NMR descriptors
#       ↓
#   Dynamic-State Language
#
# NOT USED:
#   Solution-NMR
#   composition
#   SMILES
#   evaluation endpoint Y
#
# Four language concepts:
#   1. Population balance
#   2. Distribution topology
#   3. Dynamic-state coexistence
#   4. Dynamic heterogeneity
#
# Weighted_logmean_T2_ms is retained as a numeric descriptor,
# but is deliberately NOT directly verbalized as
# "high/low mobility".
#
# ============================================================


# ============================================================
# 0. INSTALL / IMPORT
# ============================================================

import sys
import subprocess
import importlib.util


required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "sentence_transformers": "sentence-transformers",
}


missing = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]


if missing:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing,
        ]
    )


import shutil
import zipfile
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

EXPECTED_N = 43

SENTENCE_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

LOW_QUANTILE = 1.0 / 3.0
HIGH_QUANTILE = 2.0 / 3.0


# ------------------------------------------------------------
# Population balance
# ------------------------------------------------------------

DOMINANT_FRACTION_THRESHOLD = 0.60
CO_DOMINANT_DIFFERENCE = 0.10


# ------------------------------------------------------------
# Dynamic-state coexistence
# ------------------------------------------------------------

COEXISTENCE_FRACTION_THRESHOLD = 0.15


# ------------------------------------------------------------
# Number of representative materials shown in Table Sx
# ------------------------------------------------------------

N_TABLE_EXAMPLES = 4


# ============================================================
# 2. OUTPUT SETTINGS
# ============================================================

OUTPUT_DIR = Path(
    "02_Dynamic_State_Language_output"
)

ZIP_PATH = Path(
    "02_Dynamic_State_Language_output.zip"
)


if OUTPUT_DIR.exists():
    shutil.rmtree(
        OUTPUT_DIR
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


if ZIP_PATH.exists():
    ZIP_PATH.unlink()


# ============================================================
# 3. UPLOAD INPUT
# ============================================================

try:

    from google.colab import files

    print("=" * 80)
    print("Upload 01_TD_NMR_descriptors.csv")
    print("=" * 80)

    uploaded = files.upload()

    csv_files = [
        Path(name)
        for name in uploaded.keys()
        if name.lower().endswith(".csv")
    ]

except ImportError:

    csv_files = list(
        Path(".").glob("*.csv")
    )


if len(csv_files) == 0:

    raise FileNotFoundError(
        "No CSV input was found."
    )


preferred = [
    p
    for p in csv_files
    if "td_nmr_descriptors" in p.name.lower()
]


INPUT_FILE = (
    preferred[0]
    if preferred
    else csv_files[0]
)


print("\nInput file:")
print(INPUT_FILE)


# ============================================================
# 4. LOAD DESCRIPTORS
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)


REQUIRED_COLUMNS = [

    "Canonical_ID",

    "ShortFraction",

    "MidFraction",

    "LongFraction",

    "Weighted_logmean_T2_ms",

    "Width_log10T2",

    "N_detected_peaks",

    "Entropy_norm",
]


missing_columns = [
    c
    for c in REQUIRED_COLUMNS
    if c not in df.columns
]


if missing_columns:

    raise ValueError(
        "Required columns are missing:\n"
        + "\n".join(missing_columns)
    )


df = df.copy()


df["Canonical_ID"] = (
    df["Canonical_ID"]
    .astype(str)
    .str.strip()
)


if df["Canonical_ID"].duplicated().any():

    duplicates = (
        df.loc[
            df["Canonical_ID"].duplicated(
                keep=False
            ),
            "Canonical_ID",
        ]
        .tolist()
    )

    raise ValueError(
        "Duplicate Canonical_ID values detected:\n"
        + "\n".join(duplicates)
    )


for col in REQUIRED_COLUMNS[1:]:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce",
    )


if df[
    REQUIRED_COLUMNS[1:]
].isna().any().any():

    raise ValueError(
        "NaN/non-numeric descriptor values detected."
    )


fraction_sum = (
    df["ShortFraction"]
    + df["MidFraction"]
    + df["LongFraction"]
)


if not np.allclose(
    fraction_sum,
    1.0,
    atol=1e-5,
):

    raise ValueError(
        "Short + Mid + Long fractions do not sum to 1."
    )


print("\nDataset")
print("-------")
print(f"Materials: {len(df)}")


if len(df) != EXPECTED_N:

    print(
        f"WARNING: expected {EXPECTED_N} materials, "
        f"but found {len(df)}."
    )


# ============================================================
# 5. DESCRIPTIVE THRESHOLDS
# ============================================================
#
# IMPORTANT
#
# These full-dataset thresholds are used ONLY to generate
# the auditable standalone language table and SI examples.
#
# They are NOT the thresholds that should be used for the
# final repeated exploration.
#
# In 05_Final_exploration_analysis.py:
#
#   initial 30
#       ↓
#   fit q33/q67
#       ↓
#   apply same fitted rules to initial + held-out
#
# independently for every repeat.
#
# ============================================================

def quantile_thresholds(series):

    values = np.asarray(
        series,
        dtype=float,
    )

    return {
        "q33": float(
            np.quantile(
                values,
                LOW_QUANTILE,
            )
        ),

        "q67": float(
            np.quantile(
                values,
                HIGH_QUANTILE,
            )
        ),
    }


width_rules = quantile_thresholds(
    df["Width_log10T2"]
)

peak_rules = quantile_thresholds(
    df["N_detected_peaks"]
)

entropy_rules = quantile_thresholds(
    df["Entropy_norm"]
)


# ============================================================
# 6. THREE-STATE CLASSIFIER
# ============================================================

def classify_three_state(
    value,
    rules,
):

    if value <= rules["q33"]:
        return "low"

    elif value >= rules["q67"]:
        return "high"

    else:
        return "intermediate"


# ============================================================
# 7. POPULATION BALANCE
# ============================================================

def population_balance_language(
    short_fraction,
    mid_fraction,
    long_fraction,
):

    populations = {
        "short-T2": float(short_fraction),
        "intermediate-T2": float(mid_fraction),
        "long-T2": float(long_fraction),
    }


    ranked = sorted(
        populations.items(),
        key=lambda x: x[1],
        reverse=True,
    )


    top_name, top_value = ranked[0]
    second_name, second_value = ranked[1]

    difference = (
        top_value
        - second_value
    )


    if (
        top_value
        >= DOMINANT_FRACTION_THRESHOLD
    ):

        state = (
            f"{top_name} dominated"
        )

        sentence = (
            f"The relaxation population is dominated "
            f"by the {top_name} component."
        )


    elif (
        difference
        <= CO_DOMINANT_DIFFERENCE
    ):

        state = (
            f"{top_name}/{second_name} balanced"
        )

        sentence = (
            f"The relaxation population shows balanced "
            f"contributions from the {top_name} and "
            f"{second_name} components."
        )


    else:

        state = (
            f"mixed; {top_name} enriched"
        )

        sentence = (
            f"The relaxation population is mixed, "
            f"with relative enrichment of the "
            f"{top_name} component."
        )


    return state, sentence


# ============================================================
# 8. DISTRIBUTION TOPOLOGY
# ============================================================

def distribution_topology_language(
    width,
    n_peaks,
):

    width_state = classify_three_state(
        width,
        width_rules,
    )

    peak_state = classify_three_state(
        n_peaks,
        peak_rules,
    )


    if (
        width_state == "low"
        and peak_state == "low"
    ):

        state = "compact/simple"

        sentence = (
            "The relaxation distribution is compact "
            "and topologically simple."
        )


    elif (
        width_state == "high"
        and peak_state == "high"
    ):

        state = "broad/multimodal"

        sentence = (
            "The relaxation distribution is broad "
            "and contains multiple resolved dynamic populations."
        )


    elif width_state == "high":

        state = "broad"

        sentence = (
            "The relaxation distribution is broad, "
            "indicating a wide range of dynamic environments."
        )


    elif peak_state == "high":

        state = "multipeak"

        sentence = (
            "The relaxation distribution contains "
            "multiple resolved dynamic populations."
        )


    else:

        state = "intermediate"

        sentence = (
            "The relaxation distribution shows "
            "intermediate topological complexity."
        )


    return (
        state,
        sentence,
        width_state,
        peak_state,
    )


# ============================================================
# 9. DYNAMIC-STATE COEXISTENCE
# ============================================================

def coexistence_language(
    short_fraction,
    mid_fraction,
    long_fraction,
):

    populations = {
        "short-T2": float(short_fraction),
        "intermediate-T2": float(mid_fraction),
        "long-T2": float(long_fraction),
    }


    represented = [
        name
        for name, value
        in populations.items()
        if value
        >= COEXISTENCE_FRACTION_THRESHOLD
    ]


    n_states = len(
        represented
    )


    if n_states >= 3:

        state = "three-state coexistence"

        sentence = (
            "Short-, intermediate-, and long-T2 "
            "dynamic populations coexist within the material."
        )


    elif n_states == 2:

        state = "two-state coexistence"

        sentence = (
            f"The material shows coexistence of "
            f"{represented[0]} and {represented[1]} "
            f"dynamic populations."
        )


    elif n_states == 1:

        state = "single predominant state"

        sentence = (
            f"The dynamic-state distribution is concentrated "
            f"primarily in the {represented[0]} population."
        )


    else:

        state = "diffuse coexistence"

        sentence = (
            "The dynamic-state distribution is diffuse, "
            "without a strongly represented population."
        )


    return (
        state,
        sentence,
        n_states,
    )


# ============================================================
# 10. DYNAMIC HETEROGENEITY
# ============================================================

def heterogeneity_language(
    width,
    entropy,
):

    width_state = classify_three_state(
        width,
        width_rules,
    )

    entropy_state = classify_three_state(
        entropy,
        entropy_rules,
    )


    if (
        width_state == "low"
        and entropy_state == "low"
    ):

        state = "low heterogeneity"

        sentence = (
            "The material exhibits relatively low "
            "dynamic heterogeneity."
        )


    elif (
        width_state == "high"
        and entropy_state == "high"
    ):

        state = "high heterogeneity"

        sentence = (
            "The material exhibits pronounced dynamic "
            "heterogeneity across relaxation environments."
        )


    elif width_state == "high":

        state = "broad heterogeneity"

        sentence = (
            "The material shows heterogeneous dynamics "
            "primarily through a broad relaxation distribution."
        )


    elif entropy_state == "high":

        state = "distributed heterogeneity"

        sentence = (
            "The material shows heterogeneous dynamics "
            "distributed across multiple relaxation contributions."
        )


    else:

        state = "intermediate heterogeneity"

        sentence = (
            "The material exhibits an intermediate level "
            "of dynamic heterogeneity."
        )


    return (
        state,
        sentence,
        width_state,
        entropy_state,
    )


# ============================================================
# 11. GENERATE LANGUAGE
# ============================================================

language_rows = []


for _, row in df.iterrows():

    pop_state, pop_sentence = (
        population_balance_language(
            row["ShortFraction"],
            row["MidFraction"],
            row["LongFraction"],
        )
    )


    (
        topology_state,
        topology_sentence,
        width_state_topology,
        peak_state,
    ) = distribution_topology_language(
        row["Width_log10T2"],
        row["N_detected_peaks"],
    )


    (
        coexistence_state,
        coexistence_sentence,
        n_states,
    ) = coexistence_language(
        row["ShortFraction"],
        row["MidFraction"],
        row["LongFraction"],
    )


    (
        heterogeneity_state,
        heterogeneity_sentence,
        width_state_heterogeneity,
        entropy_state,
    ) = heterogeneity_language(
        row["Width_log10T2"],
        row["Entropy_norm"],
    )


    full_text = " ".join(
        [
            pop_sentence,
            topology_sentence,
            coexistence_sentence,
            heterogeneity_sentence,
        ]
    )


    language_rows.append(
        {
            "Canonical_ID":
                row["Canonical_ID"],

            "ShortFraction":
                row["ShortFraction"],

            "MidFraction":
                row["MidFraction"],

            "LongFraction":
                row["LongFraction"],

            "Weighted_logmean_T2_ms":
                row["Weighted_logmean_T2_ms"],

            "Width_log10T2":
                row["Width_log10T2"],

            "N_detected_peaks":
                row["N_detected_peaks"],

            "Entropy_norm":
                row["Entropy_norm"],

            "Population_Balance_State":
                pop_state,

            "Distribution_Topology_State":
                topology_state,

            "Dynamic_State_Coexistence_State":
                coexistence_state,

            "Dynamic_Heterogeneity_State":
                heterogeneity_state,

            "Population_Balance":
                pop_sentence,

            "Distribution_Topology":
                topology_sentence,

            "Dynamic_State_Coexistence":
                coexistence_sentence,

            "Dynamic_Heterogeneity":
                heterogeneity_sentence,

            "Topology_Width_State":
                width_state_topology,

            "Topology_Peak_State":
                peak_state,

            "Heterogeneity_Width_State":
                width_state_heterogeneity,

            "Heterogeneity_Entropy_State":
                entropy_state,

            "N_Represented_Dynamic_States":
                n_states,

            "Full_TD_Dynamic_State_Language":
                full_text,
        }
    )


language_df = pd.DataFrame(
    language_rows
)


# ============================================================
# 12. SAVE MAIN LANGUAGE
# ============================================================

LANGUAGE_PATH = (
    OUTPUT_DIR
    / "01_Dynamic_State_Language.csv"
)


main_columns = [

    "Canonical_ID",

    "Population_Balance_State",

    "Distribution_Topology_State",

    "Dynamic_State_Coexistence_State",

    "Dynamic_Heterogeneity_State",

    "Population_Balance",

    "Distribution_Topology",

    "Dynamic_State_Coexistence",

    "Dynamic_Heterogeneity",

    "Full_TD_Dynamic_State_Language",
]


language_df[
    main_columns
].to_csv(
    LANGUAGE_PATH,
    index=False,
)


# ============================================================
# 13. RULE TABLE
# ============================================================

rules_rows = [

    {
        "Language concept":
            "Population balance",

        "TD-NMR input":
            "Short-, mid-, and long-T2 fractions",

        "Rule":
            (
                "Dominant population >= 0.60; "
                "top-two difference <= 0.10 "
                "indicates balanced populations."
            ),

        "Physical meaning":
            (
                "Relative balance of relaxation "
                "populations."
            ),
    },

    {
        "Language concept":
            "Distribution topology",

        "TD-NMR input":
            "Width_log10T2 + N_detected_peaks",

        "Rule":
            (
                "Width and peak-number states are "
                "classified using q33/q67 thresholds "
                "fitted on the initial set."
            ),

        "Physical meaning":
            (
                "Compact, broad, or multimodal "
                "relaxation topology."
            ),
    },

    {
        "Language concept":
            "Dynamic-state coexistence",

        "TD-NMR input":
            "Short-, mid-, and long-T2 fractions",

        "Rule":
            (
                "A population fraction >= 0.15 is "
                "treated as meaningfully represented."
            ),

        "Physical meaning":
            (
                "Number and identity of coexisting "
                "dynamic populations."
            ),
    },

    {
        "Language concept":
            "Dynamic heterogeneity",

        "TD-NMR input":
            "Width_log10T2 + Entropy_norm",

        "Rule":
            (
                "Width and entropy states are "
                "classified using q33/q67 thresholds "
                "fitted on the initial set."
            ),

        "Physical meaning":
            (
                "Extent and distribution of "
                "dynamic heterogeneity."
            ),
    },
]


rules_df = pd.DataFrame(
    rules_rows
)


RULES_PATH = (
    OUTPUT_DIR
    / "02_Dynamic_State_Language_rules.csv"
)


rules_df.to_csv(
    RULES_PATH,
    index=False,
)


# ============================================================
# 14. SAVE AUDIT TABLE
# ============================================================

AUDIT_PATH = (
    OUTPUT_DIR
    / "03_Dynamic_State_Language_audit.csv"
)


language_df.to_csv(
    AUDIT_PATH,
    index=False,
)


# ============================================================
# 15. EMBEDDING
# ============================================================

print(
    "\nLoading sentence-transformer model..."
)


model = SentenceTransformer(
    SENTENCE_MODEL
)


texts = (
    language_df[
        "Full_TD_Dynamic_State_Language"
    ]
    .astype(str)
    .tolist()
)


print(
    "Encoding Dynamic-State Language..."
)


embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)


embeddings = np.asarray(
    embeddings,
    dtype=float,
)


embedding_columns = [
    f"TDLang_{i+1:03d}"
    for i in range(
        embeddings.shape[1]
    )
]


embedding_df = pd.DataFrame(
    embeddings,
    columns=embedding_columns,
)


embedding_df.insert(
    0,
    "Canonical_ID",
    language_df[
        "Canonical_ID"
    ].values,
)


EMBEDDING_PATH = (
    OUTPUT_DIR
    / "04_Dynamic_State_Language_embedding_384D.csv"
)


embedding_df.to_csv(
    EMBEDDING_PATH,
    index=False,
)


# ============================================================
# 16. METADATA
# ============================================================

metadata = {

    "pipeline":
        "02_Dynamic_State_Language",

    "input":
        INPUT_FILE.name,

    "n_materials":
        int(len(df)),

    "sentence_embedding_model":
        SENTENCE_MODEL,

    "embedding_dimension":
        int(embeddings.shape[1]),

    "embedding_normalized":
        True,

    "solution_NMR_used":
        False,

    "composition_used":
        False,

    "SMILES_used":
        False,

    "direct_mobility_verbalization":
        False,

    "language_concepts": [
        "Population balance",
        "Distribution topology",
        "Dynamic-state coexistence",
        "Dynamic heterogeneity",
    ],

    "final_exploration_calibration":
        (
            "q33/q67 thresholds are refitted using only "
            "the initial 30 materials within each repeat."
        ),

    "standalone_table_note":
        (
            "Full-dataset thresholds in this script are used "
            "only to generate an auditable standalone language "
            "table and representative SI examples."
        ),
}


METADATA_PATH = (
    OUTPUT_DIR
    / "05_Metadata.json"
)


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 17. SELECT REPRESENTATIVE EXAMPLES FOR TABLE Sx
# ============================================================
#
# Do NOT select examples according to Solution-NMR or
# exploration performance.
#
# Instead, choose materials that span the TD-NMR descriptor
# space.
#
# A deterministic farthest-point procedure is used.
#
# ============================================================

descriptor_cols_for_examples = [

    "ShortFraction",

    "MidFraction",

    "LongFraction",

    "Width_log10T2",

    "N_detected_peaks",

    "Entropy_norm",
]


X = (
    language_df[
        descriptor_cols_for_examples
    ]
    .astype(float)
    .to_numpy()
)


X_mean = X.mean(
    axis=0
)


X_std = X.std(
    axis=0,
    ddof=0,
)


X_std[
    X_std < 1e-12
] = 1.0


Xz = (
    X - X_mean
) / X_std


# Start from material closest to the multivariate center
center_distance = np.sqrt(
    np.sum(
        Xz ** 2,
        axis=1,
    )
)


selected = [
    int(
        np.argmin(
            center_distance
        )
    )
]


while len(selected) < min(
    N_TABLE_EXAMPLES,
    len(language_df),
):

    candidate_indices = [
        i
        for i in range(
            len(language_df)
        )
        if i not in selected
    ]


    best_index = None
    best_distance = -np.inf


    for candidate in candidate_indices:

        distances = [
            np.linalg.norm(
                Xz[candidate]
                - Xz[s]
            )
            for s in selected
        ]


        minimum_distance = min(
            distances
        )


        if minimum_distance > best_distance:

            best_distance = (
                minimum_distance
            )

            best_index = (
                candidate
            )


    selected.append(
        best_index
    )


examples_df = (
    language_df
    .iloc[selected]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 18. COMPACT EXAMPLE TABLE
# ============================================================

examples_compact_df = pd.DataFrame(
    {
        "Material":
            examples_df[
                "Canonical_ID"
            ],

        "Short/Mid/Long":
            examples_df.apply(
                lambda r:
                (
                    f"{r['ShortFraction']:.2f}/"
                    f"{r['MidFraction']:.2f}/"
                    f"{r['LongFraction']:.2f}"
                ),
                axis=1,
            ),

        "Width":
            examples_df[
                "Width_log10T2"
            ]
            .map(
                lambda x:
                f"{x:.2f}"
            ),

        "Peaks":
            examples_df[
                "N_detected_peaks"
            ]
            .map(
                lambda x:
                f"{int(round(x))}"
            ),

        "Entropy":
            examples_df[
                "Entropy_norm"
            ]
            .map(
                lambda x:
                f"{x:.2f}"
            ),

        "Population balance":
            examples_df[
                "Population_Balance_State"
            ],

        "Topology":
            examples_df[
                "Distribution_Topology_State"
            ],

        "Coexistence":
            examples_df[
                "Dynamic_State_Coexistence_State"
            ],

        "Heterogeneity":
            examples_df[
                "Dynamic_Heterogeneity_State"
            ],
    }
)


# ============================================================
# 19. TABLE Sx EXCEL
# ============================================================

TABLE_XLSX_PATH = (
    OUTPUT_DIR
    / "06_Table_Sx_Dynamic_State_Language.xlsx"
)


with pd.ExcelWriter(
    TABLE_XLSX_PATH,
    engine="openpyxl",
) as writer:

    rules_df.to_excel(
        writer,
        sheet_name="Language_rules",
        index=False,
    )

    examples_compact_df.to_excel(
        writer,
        sheet_name="Representative_examples",
        index=False,
    )


# ============================================================
# 20. HELPER FOR A4 TABLE
# ============================================================

def wrap_text(
    value,
    width,
):

    return "\n".join(
        textwrap.wrap(
            str(value),
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )
    )


# ============================================================
# 21. GENERATE A4 LANDSCAPE TABLE Sx
# ============================================================
#
# A4 landscape:
# 11.69 × 8.27 inch
#
# Designed as ONE SI page.
#
# ============================================================

fig = plt.figure(
    figsize=(
        11.69,
        8.27,
    )
)


# ------------------------------------------------------------
# Main title
# ------------------------------------------------------------

fig.text(
    0.04,
    0.955,

    "Table Sx. Transformation of TD-NMR descriptors into "
    "experimentally grounded Dynamic-State Language",

    fontsize=12,
    fontweight="bold",
    va="top",
)


# ------------------------------------------------------------
# Explanatory note
# ------------------------------------------------------------

fig.text(
    0.04,
    0.905,

    "Dynamic-State Language is constructed exclusively from "
    "D2O-swollen CPMG-derived TD-NMR descriptors. "
    "Solution-NMR responses are not used in representation construction. "
    "For repeated exploration, q33/q67 thresholds are fitted using "
    "only the initial training set within each repeat.",

    fontsize=8.2,
    va="top",
    wrap=True,
)


# ============================================================
# 21A. TOP TABLE — LANGUAGE RULES
# ============================================================

ax_top = fig.add_axes(
    [
        0.04,
        0.52,
        0.92,
        0.33,
    ]
)

ax_top.axis(
    "off"
)


top_headers = [

    "Language concept",

    "TD-NMR input",

    "Transformation rule",

    "Physical interpretation",
]


top_rows = []


for _, r in rules_df.iterrows():

    top_rows.append(
        [
            wrap_text(
                r[
                    "Language concept"
                ],
                22,
            ),

            wrap_text(
                r[
                    "TD-NMR input"
                ],
                26,
            ),

            wrap_text(
                r[
                    "Rule"
                ],
                44,
            ),

            wrap_text(
                r[
                    "Physical meaning"
                ],
                34,
            ),
        ]
    )


table_top = ax_top.table(

    cellText=top_rows,

    colLabels=top_headers,

    colWidths=[
        0.18,
        0.22,
        0.36,
        0.24,
    ],

    cellLoc="left",

    colLoc="left",

    bbox=[
        0,
        0,
        1,
        1,
    ],
)


table_top.auto_set_font_size(
    False
)

table_top.set_fontsize(
    7.4
)


for (
    row,
    col
), cell in table_top.get_celld().items():

    cell.set_edgecolor(
        "0.70"
    )

    cell.set_linewidth(
        0.6
    )

    cell.PAD = 0.035

    if row == 0:

        cell.set_text_props(
            weight="bold"
        )

        cell.set_facecolor(
            "0.92"
        )

    else:

        if col == 0:

            cell.set_text_props(
                weight="bold"
            )


# ============================================================
# 21B. SECTION LABEL
# ============================================================

fig.text(
    0.04,
    0.475,

    "Representative examples spanning the TD-NMR descriptor space",

    fontsize=9.5,
    fontweight="bold",
    va="top",
)


fig.text(
    0.04,
    0.445,

    "Representative materials were selected using TD-NMR "
    "descriptor-space diversity only; no Solution-NMR response "
    "or exploration performance was used for example selection.",

    fontsize=7.7,
    va="top",
)


# ============================================================
# 21C. BOTTOM TABLE — EXAMPLES
# ============================================================

ax_bottom = fig.add_axes(
    [
        0.04,
        0.16,
        0.92,
        0.25,
    ]
)

ax_bottom.axis(
    "off"
)


bottom_headers = [

    "Material",

    "Short/Mid/Long",

    "Width",

    "Peaks",

    "Entropy",

    "Population\nbalance",

    "Topology",

    "Coexistence",

    "Heterogeneity",
]


bottom_rows = []


for _, r in examples_compact_df.iterrows():

    bottom_rows.append(
        [
            wrap_text(
                r["Material"],
                18,
            ),

            r[
                "Short/Mid/Long"
            ],

            r[
                "Width"
            ],

            r[
                "Peaks"
            ],

            r[
                "Entropy"
            ],

            wrap_text(
                r[
                    "Population balance"
                ],
                18,
            ),

            wrap_text(
                r[
                    "Topology"
                ],
                16,
            ),

            wrap_text(
                r[
                    "Coexistence"
                ],
                17,
            ),

            wrap_text(
                r[
                    "Heterogeneity"
                ],
                18,
            ),
        ]
    )


table_bottom = ax_bottom.table(

    cellText=bottom_rows,

    colLabels=bottom_headers,

    colWidths=[
        0.15,
        0.11,
        0.06,
        0.055,
        0.065,
        0.14,
        0.11,
        0.145,
        0.16,
    ],

    cellLoc="center",

    colLoc="center",

    bbox=[
        0,
        0,
        1,
        1,
    ],
)


table_bottom.auto_set_font_size(
    False
)

table_bottom.set_fontsize(
    6.9
)


for (
    row,
    col
), cell in table_bottom.get_celld().items():

    cell.set_edgecolor(
        "0.70"
    )

    cell.set_linewidth(
        0.6
    )

    cell.PAD = 0.025

    if row == 0:

        cell.set_text_props(
            weight="bold"
        )

        cell.set_facecolor(
            "0.92"
        )

    elif col == 0:

        cell.set_text_props(
            weight="bold"
        )


# ============================================================
# 21D. FOOTNOTE
# ============================================================

fig.text(
    0.04,
    0.105,

    "Short/Mid/Long: fractional populations assigned to "
    "T2 < 1 ms, 1 <= T2 < 30 ms, and T2 >= 30 ms, respectively. "
    "Width: standard deviation of log10(T2). "
    "Peaks: number of detected ILT peaks. "
    "Entropy: normalized entropy of the ILT distribution.",

    fontsize=7.1,
    va="top",
    wrap=True,
)


fig.text(
    0.04,
    0.055,

    "Weighted mean T2 is retained in the numeric TD-NMR "
    "representation but is deliberately not translated directly "
    "into a high/low mobility term in Dynamic-State Language.",

    fontsize=7.1,
    va="top",
    wrap=True,
)


# ============================================================
# 22. SAVE TABLE Sx
# ============================================================

TABLE_PDF_PATH = (
    OUTPUT_DIR
    / "Table_Sx_Dynamic_State_Language.pdf"
)


TABLE_PNG_PATH = (
    OUTPUT_DIR
    / "Table_Sx_Dynamic_State_Language.png"
)


plt.savefig(
    TABLE_PDF_PATH,
    bbox_inches="tight",
)


plt.savefig(
    TABLE_PNG_PATH,
    dpi=600,
    bbox_inches="tight",
)


plt.close(
    fig
)


# ============================================================
# 23. VERIFY OUTPUTS
# ============================================================

EXPECTED_OUTPUTS = [

    LANGUAGE_PATH,

    RULES_PATH,

    AUDIT_PATH,

    EMBEDDING_PATH,

    METADATA_PATH,

    TABLE_XLSX_PATH,

    TABLE_PDF_PATH,

    TABLE_PNG_PATH,
]


for path in EXPECTED_OUTPUTS:

    if not path.exists():

        raise FileNotFoundError(
            f"Expected output was not generated: {path}"
        )


# ============================================================
# 24. ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:

    for path in EXPECTED_OUTPUTS:

        z.write(
            path,
            arcname=path.name,
        )


# ============================================================
# 25. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "02_Dynamic_State_Language completed successfully."
)

print(
    "=" * 80
)


print(
    "\nGenerated files:"
)


for path in EXPECTED_OUTPUTS:

    print(
        "  -",
        path.name,
    )


print(
    "\nRepresentative materials used in Table Sx:"
)


for material in (
    examples_compact_df[
        "Material"
    ]
):

    print(
        "  -",
        material
    )


print(
    "\nTable Sx:"
)

print(
    "  A4 landscape"
)

print(
    "  One page"
)

print(
    "  Top: transformation rules"
)

print(
    "  Bottom: four representative examples"
)


print(
    "\nLeakage separation:"
)

print(
    "  TD-NMR             : USED"
)

print(
    "  Solution-NMR       : NOT USED"
)

print(
    "  Composition/SMILES : NOT USED"
)

print(
    "  Evaluation Y       : NOT USED"
)


print(
    "\nIMPORTANT:"
)

print(
    "  Table Sx examples use the standalone descriptive "
    "language table."
)

print(
    "  In the final 200-repeat exploration, q33/q67 "
    "thresholds must be fitted independently using only "
    "the initial 30 materials in each repeat."
)


print(
    "\nZIP:"
)

print(
    ZIP_PATH
)


# ============================================================
# 26. AUTOMATIC DOWNLOAD
# ============================================================

try:

    from google.colab import files

    files.download(
        str(
            ZIP_PATH
        )
    )

except ImportError:

    print(
        "\nNot running in Google Colab. "
        "The ZIP remains in the current directory."
    )

Upload 01_TD_NMR_descriptors.csv


Saving 01_TD_NMR_descriptors.csv to 01_TD_NMR_descriptors (1).csv

Input file:
01_TD_NMR_descriptors (1).csv

Dataset
-------
Materials: 43

Loading sentence-transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding Dynamic-State Language...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


02_Dynamic_State_Language completed successfully.

Generated files:
  - 01_Dynamic_State_Language.csv
  - 02_Dynamic_State_Language_rules.csv
  - 03_Dynamic_State_Language_audit.csv
  - 04_Dynamic_State_Language_embedding_384D.csv
  - 05_Metadata.json
  - 06_Table_Sx_Dynamic_State_Language.xlsx
  - Table_Sx_Dynamic_State_Language.pdf
  - Table_Sx_Dynamic_State_Language.png

Representative materials used in Table Sx:
  - pSSA_TFEMA_DICL
  - HEA_HMA_TECL
  - VBA_HMA_DICL
  - NIPAM_HMA_TECL

Table Sx:
  A4 landscape
  One page
  Top: transformation rules
  Bottom: four representative examples

Leakage separation:
  TD-NMR             : USED
  Solution-NMR       : NOT USED
  Composition/SMILES : NOT USED
  Evaluation Y       : NOT USED

IMPORTANT:
  Table Sx examples use the standalone descriptive language table.
  In the final 200-repeat exploration, q33/q67 thresholds must be fitted independently using only the initial 30 materials in each repeat.

ZIP:
02_Dynamic_State_Language_output.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>